# Midstance Entropy — Per-Step Bar Graph on GPS Route

Each detected step is drawn as a **rectangular bar** on the map:

- Bar **base** sits on the GPS route at the step's timestamp position  
- Bar **height** = evenness score (0–1)  
- Bar **direction** = perpendicular to the route at that point  
- **Right foot** bars extend to the right of travel → blue  
- **Left foot** bars extend to the left of travel → orange  

The result is a **back-to-back bar chart wrapped around the GPS path** —
taller bars = more even pressure distribution across the 12 sensors.

```
   ██  █  ███ ██   ← right foot bars (evenness score)
   ██  █  ███ ██
───────────────── GPS route
   ██ ██   █  ██
   ██ ██   █  ██   ← left foot bars
```

**Output:** `entropy_bar_map.html`

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from scipy.stats import entropy as scipy_entropy
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import folium
import warnings
warnings.filterwarnings('ignore')

print(f'folium {folium.__version__}')

folium 0.20.0


## Cell 2 — Configuration

In [2]:
CSV_PATH = 'Kristian_dry_0425.csv'
GPX_PATH = 'activity_22645980458.gpx'

PRESSURE_COLS     = [f'pressure_{i:02d}' for i in range(1, 13)]
MAX_ENTROPY       = np.log2(12)
SAMPLE_RATE_MS    = 4
GPS_TOL_MS        = 5000   # ±5 s GPS match tolerance

# ── Stance detection ──────────────────────────────────────────────────────────
STANCE_PERCENTILE = 15
MIN_STANCE_SAMP   = 20
MAX_STANCE_SAMP   = 500
MS_START, MS_END  = 1/3, 2/3

# ── Step grouping ─────────────────────────────────────────────────────────────
# Steps are grouped per foot before drawing. Each group of STEP_GROUP consecutive
# steps is averaged into a single bar.
# Set to 1 to restore one-bar-per-step behaviour.
STEP_GROUP = 10

# ── Bar geometry ──────────────────────────────────────────────────────────────
# Max bar height at evenness = 1.0  (~22 m on the ground)
BAR_MAX_HEIGHT = 0.00020
# Half-width of each bar along the route direction.
# With grouped bars, increase this so bars are visually wider and easier to read.
BAR_HALF_WIDTH = 0.00008   # ~9 m — wider than single-step bars

# ── Colours ───────────────────────────────────────────────────────────────────
RIGHT_COLOR  = '#1F77B4'   # blue
LEFT_COLOR   = '#FF7F0E'   # orange
BASE_COLOR   = '#555555'

print('Config ready.')
print(f'  Step group size: {STEP_GROUP} steps per bar')
print(f'  Bar max height : {BAR_MAX_HEIGHT * 111000:.0f} m at evenness = 1.0')
print(f'  Bar half-width : {BAR_HALF_WIDTH * 111000:.1f} m along route')

Config ready.
  Step group size: 10 steps per bar
  Bar max height : 22 m at evenness = 1.0
  Bar half-width : 8.9 m along route


## Cell 3 — Load Data

In [3]:
raw = pd.read_csv(CSV_PATH, low_memory=False)
raw = raw[raw['corrupt'] == 0].copy()
raw['timestamp']      = pd.to_numeric(raw['timestamp'], errors='coerce')
raw['total_pressure'] = raw[PRESSURE_COLS].sum(axis=1)

def parse_gpx(path):
    tree = ET.parse(path)
    root = tree.getroot()
    ns   = {'gpx': 'http://www.topografix.com/GPX/1/1'}
    rows = []
    for trkpt in root.findall('.//gpx:trkpt', ns):
        dt = pd.to_datetime(trkpt.find('gpx:time', ns).text.strip(), utc=True)
        rows.append({'lat': float(trkpt.get('lat')),
                     'lon': float(trkpt.get('lon')),
                     'unix_ms': int(dt.value // 1_000_000)})
    return pd.DataFrame(rows)

gpx_df = parse_gpx(GPX_PATH)
print(f'Insole : {len(raw):,} rows | soles {sorted(raw.sole_id.unique())}')
print(f'GPX    : {len(gpx_df)} trackpoints')

Insole : 51,267 rows | soles [np.int64(1), np.int64(2)]
GPX    : 440 trackpoints


## Cell 4 — Midstance Entropy Extraction

In [4]:
def detect_stances(sdf):
    sig = sdf['total_pressure'].values
    thr = np.percentile(sig, STANCE_PERCENTILE)
    in_stance = sig >= thr
    windows, i = [], 0
    while i < len(in_stance):
        if in_stance[i]:
            start = i
            while i < len(in_stance) and in_stance[i]: i += 1
            dur = i - start
            if MIN_STANCE_SAMP <= dur <= MAX_STANCE_SAMP:
                windows.append({'start': start, 'end': i, 'duration': dur})
        else:
            i += 1
    return windows


def extract_entropy(sdf, sole_id):
    foot = 'RIGHT' if sole_id == 1 else 'LEFT'
    windows = detect_stances(sdf)
    records = []
    for step_idx, win in enumerate(windows):
        s, e, d = win['start'], win['end'], win['duration']
        ms_s = s + int(d * MS_START)
        ms_e = s + int(d * MS_END)
        if ms_e <= ms_s: continue
        ms_slice = sdf.iloc[ms_s:ms_e]
        mp = ms_slice[PRESSURE_COLS].mean().values.astype(float)
        total = mp.sum()
        if total < 10: continue
        p_norm = np.clip(mp / total, 1e-10, 1.0)
        H = scipy_entropy(p_norm, base=2)
        evenness = H / MAX_ENTROPY
        centre_ts = int(sdf.iloc[(ms_s + ms_e) // 2]['timestamp'])
        records.append({
            'step_idx': step_idx, 'sole_id': sole_id, 'foot': foot,
            'evenness_score': evenness, 'H_spatial': H,
            'unevenness_score': 1 - evenness,
            'stance_dur_ms': d * SAMPLE_RATE_MS,
            'contact_timestamp': centre_ts,
        })
    return pd.DataFrame(records)


all_steps = []
for sole_id in [1, 2]:
    sdf = raw[raw['sole_id'] == sole_id].sort_values('timestamp').reset_index(drop=True)
    feat = extract_entropy(sdf, sole_id)
    all_steps.append(feat)
    foot = 'RIGHT' if sole_id == 1 else 'LEFT'
    print(f'{foot}: {len(feat)} steps | '
          f'evenness {feat["evenness_score"].mean():.4f} ± {feat["evenness_score"].std():.4f}')

steps = pd.concat(all_steps, ignore_index=True)
print(f'Total: {len(steps)} steps')

RIGHT: 340 steps | evenness 0.9875 ± 0.0043
LEFT: 290 steps | evenness 0.9906 ± 0.0023
Total: 630 steps


## Cell 5 — GPS Matching

In [5]:
matched = pd.merge_asof(
    steps.sort_values('contact_timestamp'),
    gpx_df.sort_values('unix_ms')[['unix_ms','lat','lon']]
          .rename(columns={'unix_ms': 'contact_timestamp'}),
    on='contact_timestamp',
    direction='nearest',
    tolerance=GPS_TOL_MS,
)
steps = matched.sort_values(['sole_id','step_idx']).reset_index(drop=True)
steps['walk_time_s'] = (steps['contact_timestamp'] - steps['contact_timestamp'].min()) / 1000

ok  = steps.dropna(subset=['lat','lon'])
bad = steps[steps['lat'].isna()]
print(f'Steps matched to GPS : {len(ok)}')
print(f'Steps outside GPS    : {len(bad)}')

Steps matched to GPS : 630
Steps outside GPS    : 0


## Cell 5b — Group Steps

Consecutive steps (per foot, in walk order) are grouped into windows of
`STEP_GROUP` size. Each group is collapsed to a single row by taking:
- **mean** evenness score → bar height
- **median** contact timestamp → GPS anchor point for the bar
- **min/max** step indices → for the popup label

In [6]:
def group_steps(df, group_size):
    """
    Group consecutive steps (per foot) into windows of `group_size`.
    Returns one row per group with averaged metrics and median GPS anchor.
    """
    rows = []
    for foot, fdf in df.groupby('foot', sort=False):
        fdf = fdf.sort_values('contact_timestamp').reset_index(drop=True)
        for g_start in range(0, len(fdf), group_size):
            grp = fdf.iloc[g_start : g_start + group_size]
            rows.append({
                'foot'            : foot,
                'group_id'        : g_start // group_size,
                'n_steps'         : len(grp),
                'step_range'      : f"{int(grp['step_idx'].min())}–{int(grp['step_idx'].max())}",
                'evenness_score'  : grp['evenness_score'].mean(),
                'evenness_std'    : grp['evenness_score'].std(),
                'H_spatial'       : grp['H_spatial'].mean(),
                'unevenness_score': grp['unevenness_score'].mean(),
                'stance_dur_ms'   : grp['stance_dur_ms'].mean(),
                # Use the median timestamp as the GPS anchor
                'contact_timestamp': int(grp['contact_timestamp'].median()),
                # Carry the matched GPS coords from the median step
                'lat'             : grp.loc[grp['contact_timestamp']
                                            == grp['contact_timestamp'].median()
                                            ].iloc[0]['lat']
                                   if (grp['contact_timestamp']
                                       == grp['contact_timestamp'].median()).any()
                                   else grp['lat'].dropna().median(),
                'lon'             : grp.loc[grp['contact_timestamp']
                                            == grp['contact_timestamp'].median()
                                            ].iloc[0]['lon']
                                   if (grp['contact_timestamp']
                                       == grp['contact_timestamp'].median()).any()
                                   else grp['lon'].dropna().median(),
                'walk_time_s'     : grp['walk_time_s'].mean(),
            })
    return pd.DataFrame(rows)


grouped = group_steps(steps, STEP_GROUP)

print(f'Steps  → {len(steps)}  individual steps')
print(f'Groups → {len(grouped)} bars  ({STEP_GROUP} steps per bar)')
print()
for foot in ['RIGHT', 'LEFT']:
    g = grouped[grouped['foot'] == foot]
    print(f'  {foot}: {len(g)} bars | '
          f'mean evenness {g["evenness_score"].mean():.4f} ± {g["evenness_score"].std():.4f}')

Steps  → 630  individual steps
Groups → 63 bars  (10 steps per bar)

  RIGHT: 34 bars | mean evenness 0.9875 ± 0.0027
  LEFT: 29 bars | mean evenness 0.9906 ± 0.0017


## Cell 6 — Bar Geometry Functions

Each bar is a `Polygon` with 4 corners:

```
  D ─────── C   ← top edge  (height = evenness × BAR_MAX_HEIGHT)
  │         │
  A ─────── B   ← base on route (zero reference)
```

The bar is oriented **perpendicular to the route direction** at the step's GPS
point.  Right foot bars go to the right of travel, left foot to the left.

In [7]:
def tan_perp_units(lats, lons, i):
    """Return (tangent, perpendicular) unit vectors at index i."""
    n = len(lats)
    lat_scale = np.cos(np.radians(lats[i]))
    if 0 < i < n - 1:
        dlat = lats[i+1] - lats[i-1]
        dlon = (lons[i+1] - lons[i-1]) * lat_scale
    elif i == 0:
        dlat = lats[1] - lats[0]
        dlon = (lons[1] - lons[0]) * lat_scale
    else:
        dlat = lats[-1] - lats[-2]
        dlon = (lons[-1] - lons[-2]) * lat_scale
    mag = np.sqrt(dlat**2 + dlon**2) + 1e-12
    # tangent unit vector
    t_lat = dlat / mag
    t_lon = dlon / mag / lat_scale
    # perpendicular (90° CCW rotation)
    p_lat = -dlon / mag
    p_lon =  dlat / mag / lat_scale
    return (t_lat, t_lon), (p_lat, p_lon)


def make_bar(base_lat, base_lon, tan, perp, height, half_width, side):
    """
    4-corner polygon for one bar.
    side = +1 (right of travel) or -1 (left of travel).
    """
    t_lat, t_lon = tan
    p_lat, p_lon = perp
    h = float(height) * side
    w = float(half_width)

    A = (base_lat - t_lat * w,           base_lon - t_lon * w          )  # base left
    B = (base_lat + t_lat * w,           base_lon + t_lon * w          )  # base right
    C = (base_lat + t_lat * w + p_lat*h, base_lon + t_lon * w + p_lon*h)  # top  right
    D = (base_lat - t_lat * w + p_lat*h, base_lon - t_lon * w + p_lon*h)  # top  left
    return [A, B, C, D]


# Pre-compute tangent/perp at every GPX point
gpx_lats = gpx_df['lat'].values
gpx_lons = gpx_df['lon'].values
tan_perp  = [tan_perp_units(gpx_lats, gpx_lons, i) for i in range(len(gpx_df))]

print('Bar geometry functions ready ✓')

Bar geometry functions ready ✓


## Cell 7 — Build Folium Map

In [8]:
centre_lat = gpx_df['lat'].mean()
centre_lon = gpx_df['lon'].mean()

m = folium.Map(
    location=[centre_lat, centre_lon],
    zoom_start=17,
    tiles='CartoDB positron',
)

# GPS baseline route
folium.PolyLine(
    list(zip(gpx_df['lat'], gpx_df['lon'])),
    color=BASE_COLOR, weight=2, opacity=0.6,
    tooltip='GPS route'
).add_to(m)

# One FeatureGroup per foot — iterate over GROUPED data
for foot, side, color in [('RIGHT', +1, RIGHT_COLOR), ('LEFT', -1, LEFT_COLOR)]:
    fg = folium.FeatureGroup(name=f'{foot} foot ({STEP_GROUP}-step groups)', show=True)
    foot_groups = grouped[(grouped['foot'] == foot) & grouped['lat'].notna()]

    for _, row in foot_groups.iterrows():
        # Nearest GPX index for direction vectors
        gidx = int((gpx_df['unix_ms'] - row['contact_timestamp']).abs().argmin())
        tan, perp = tan_perp[gidx]

        height = row['evenness_score'] * BAR_MAX_HEIGHT
        corners = make_bar(
            row['lat'], row['lon'], tan, perp,
            height=height, half_width=BAR_HALF_WIDTH, side=side
        )

        popup_html = (
            f"<b>{foot} — Steps {row['step_range']}</b><br>"
            f"Group size  : {int(row['n_steps'])} steps<br>"
            f"Mean evenness : <b>{row['evenness_score']:.4f}</b><br>"
            f"Std evenness  : {row['evenness_std']:.4f}<br>"
            f"Mean H spatial: {row['H_spatial']:.4f} bits<br>"
            f"Mean unevenness: {row['unevenness_score']:.4f}<br>"
            f"Mean stance   : {row['stance_dur_ms']:.0f} ms<br>"
            f"Walk time     : {row['walk_time_s']:.1f} s"
        )

        folium.Polygon(
            locations=corners,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.75,
            weight=1,
            opacity=0.9,
            popup=folium.Popup(popup_html, max_width=240),
            tooltip=(
                f"{foot} steps {row['step_range']} | "
                f"mean evenness={row['evenness_score']:.4f}"
            ),
        ).add_to(fg)

    fg.add_to(m)

# Start / end markers
folium.Marker([gpx_df['lat'].iloc[0],  gpx_df['lon'].iloc[0]],
              icon=folium.Icon(color='green', icon='play', prefix='fa'),
              tooltip='Walk start').add_to(m)
folium.Marker([gpx_df['lat'].iloc[-1], gpx_df['lon'].iloc[-1]],
              icon=folium.Icon(color='red', icon='stop', prefix='fa'),
              tooltip='Walk end').add_to(m)

# Legend
legend_html = f"""
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
    background:white; padding:12px 16px; border-radius:8px;
    box-shadow:0 2px 8px rgba(0,0,0,0.25); font-family:Arial; font-size:12px;">
  <b style='font-size:13px;'>Midstance Evenness Score</b><br>
  <span style='color:{BASE_COLOR};'>&#9644;</span> GPS route (zero baseline)<br>
  <span style='color:{RIGHT_COLOR};'>&#9646;</span> Right foot bars (right of travel)<br>
  <span style='color:{LEFT_COLOR};'>&#9646;</span> Left foot bars (left of travel)<br>
  <hr style='margin:6px 0;'>
  Each bar = mean of <b>{STEP_GROUP} steps</b><br>
  Bar height ∝ mean evenness score (0–1)<br>
  Max height: {BAR_MAX_HEIGHT*111000:.0f} m at score = 1.0<br>
  <i style='font-size:11px;'>Click bars for group details + std</i><br>
  <i style='font-size:11px;'>Toggle feet via layer control (top-right)</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

m.save('entropy_bar_map.html')
print(f'Saved → entropy_bar_map.html  ({len(grouped)} bars total)')
m

Saved → entropy_bar_map.html  (63 bars total)
